# 365 Probabilidades - Dia #160
## Qual a probabilidade de a cor da pele mudar o tratamento que uma pessoa recebe?

**Tipo:** Comportamental  
**Data de publicação:** 2026-11-20  
**Ferramenta:** Python  
**Decisão analisada:** Quanto do preconceito dá para medir, e em que lugares ele aparece?  
**Hashtag:** #365Probabilidades #Dia160

---

### 📖 A História

Hoje é 20 de novembro, Dia da Consciência Negra.

É um dia em que se fala muito de preconceito, e preconceito é difícil de medir. Quase ninguém diz que tem, e quem sofre nem sempre consegue provar.

Existem duas formas de chegar perto. Uma é perguntar às pessoas o que elas vivem. A outra é comparar duas pessoas iguais em tudo o que se consegue igualar e ver se o tratamento muda com a cor.

Fui ver o que cada uma mostra no Brasil, em quatro lugares.

---

### 📚 O Conceito

Uma pesquisa de opinião mede o relato. Ela mostra o que as pessoas percebem no dia a dia, e não consegue separar o que aconteceu de como foi percebido.

Os estudos deste dia medem o comportamento, cada um de um jeito.

- **Na escola**, cada aluno tem duas notas de matemática. Uma é dada pelo professor, que conhece o rosto e o nome. A outra vem de uma prova corrigida sem identificação.
- **No currículo**, o pesquisador envia currículos fictícios iguais, com a mesma foto, e altera só o tom de pele. Como o tom é sorteado, a diferença no retorno é efeito da cor. É o único dos quatro desenhos que permite falar em causa.
- **Na empresa**, registros de emprego mostram quanto cada empresa paga e quem trabalha nela. Os autores separam a parte da diferença de salário ligada à qualificação da parte que a qualificação não explica.
- **Na publicidade**, o mesmo anúncio vai ao ar com a mesma foto em pele clara e em pele escura, e conta-se a reação do público.

---

### 🧮 O Modelo

São cinco fontes e sete contas próprias.

1. **O que sobra da diferença de nota** quando o desempenho e o comportamento são igualados.
2. **A diferença em alunos**, com o coeficiente traduzido em alunos a cada mil (200 mil sorteios, semente 42).
3. **O professor que conhece o aluno**, com o teste da diferença entre os dois grupos.
4. **O que as pessoas relatam**, com o fator ×0,80 e o intervalo publicado.
5. **O currículo**, com distribuição Beta para cada taxa de retorno e a probabilidade de o retorno ser menor com pele escura.
6. **A empresa**, com a diferença de salário traduzida em reais para cada R$ 1.000.
7. **A publicidade**, com a razão entre as taxas de curtida e o seu intervalo.

**Fontes:**
- Botelho, F., Madeira, R. A. & Rangel, M. A. (2015). Racial Discrimination in Grading: Evidence from Brazil. *American Economic Journal: Applied Economics*, 7(4), 37-52. Rede estadual de São Paulo, oitavo ano, 2010, 352.881 alunos em 11.475 turmas. **Números da versão de trabalho de janeiro de 2014. A versão publicada não foi lida.**
- Dias, F. A. (2017). *How Skin Color, Class Status, and Gender Shape Labor Market Outcomes in Brazil*. Tese de doutorado, Universidade da Califórnia em Berkeley. 1.200 currículos fictícios com foto, São Paulo e Rio de Janeiro, 2014-2015. **A versão em artigo (Dias, 2020, *Research in Social Stratification and Mobility*, 65, 100477) informa 820 currículos e 88 retornos. Só o resumo do artigo foi lido, e os números usados aqui são os da tese.**
- Gerard, F., Lagos, L., Severnini, E. & Card, D. (2021). Assortative Matching or Exclusionary Hiring? The Impact of Employment and Pay Policies on Racial Wage Differences in Brazil. *American Economic Review*, 111(10), 3418-3457. RAIS, Sudeste, 2002 a 2014, cerca de 44 milhões de observações só entre homens brancos. **Números da versão de trabalho de 2018 (IZA DP 11923). A versão publicada não foi lida.**
- Sehgal, N. K. R. & Svirsky, D. (2024). Race Discrimination in Internet Advertising: Evidence From a Field Experiment. arXiv:2412.14307. **Texto sem revisão por pares.** No Brasil, 131.219 visualizações e 18.963 curtidas, em réplica exploratória.
- Vital Strategies, Umane, Universidade Federal de Pelotas e Ministério da Igualdade Racial (2025). *Mais Dados Mais Saúde: experiência de discriminação cotidiana pela população brasileira*. Questionário online, 2.458 respostas, agosto e setembro de 2024, amostra não probabilística com pesos do Censo 2022. Lido no relatório completo.

**Nota metodológica sobre o fator ×0,80.** Aplicado só nas proporções da pesquisa, que são autorrelato, sempre com o número original ao lado. Não se aplica às notas, aos currículos, aos salários nem aos anúncios, que são registros ou experimentos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

SEMENTE = 42
N_SIM = 200_000
rng = np.random.default_rng(SEMENTE)

VERMELHO = '#c0392b'
DOURADO = '#c8a84b'
VERDE = '#1a5f5a'
CINZA = '#7f8c8d'

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---
# Botelho, Madeira & Rangel (2015), American Economic Journal: Applied Economics 7(4), 37-52
# Números lidos na versão de trabalho de janeiro de 2014 ("Discrimination Goes to School")
# Rede estadual de São Paulo, oitavo ano, 2010. Nota de matemática de 0 a 10, com 5 para passar.

n_alunos = 352_881
n_turmas = 11_475

# Diferença entre alunos negros e brancos da mesma turma: (coeficiente, erro padrão)
# Nota do professor, em pontos
nota_bruta = -0.556
nota_controlada = (-0.084, 0.008)      # mesmo desempenho na prova cega e mesmo comportamento
nota_com_passado = (-0.056, 0.007)     # mais as notas de anos anteriores
nota_instrumental = (-0.035, 0.008)    # corrigindo o erro de medida da prova

# Ter nota para passar (5 ou mais), em pontos percentuais
passar_bruta = -5.364
passar_controlada = (-1.018, 0.198)
passar_com_passado = (-0.767, 0.196)
passar_instrumental = (-0.577, 0.204)
p_passar_negros = 85.21                # % de alunos negros com nota para passar, citado no texto

# Ficar acima da média da turma, em pontos percentuais
media_bruta = -12.432
media_controlada = (-2.495, 0.268)
media_com_passado = (-1.738, 0.259)
media_instrumental = (-1.138, 0.282)

# Onde a diferença muda (nota, especificação instrumental)
nao_conhecia = (-0.041, 0.009)         # professor não conhecia o aluno
ja_conhecia = (-0.011, 0.016)          # professor já conhecia o aluno
prof_branco = (-0.036, 0.009)
prof_preto_pardo = (-0.035, 0.019)
meninos = (-0.048, 0.011)
meninas = (-0.019, 0.013)

# Vital Strategies, Umane, UFPel e Ministério da Igualdade Racial (2025)
# "Mais Dados Mais Saúde: experiência de discriminação cotidiana pela população brasileira"
# Questionário online, 2.458 respostas, agosto e setembro de 2024, amostra não probabilística com pesos
n_pesquisa = 2458
# "Ser tratado com menos respeito": (estimativa %, limite inferior, limite superior do IC 95% publicado)
respeito_alguma_vez = {          # raramente, frequentemente ou sempre
    'Brancos': (46.5, 36.3, 56.6),
    'Pardos': (60.1, 43.0, 77.2),
    'Pretos': (92.5, 84.9, 100.0),
}
respeito_frequente = {           # frequentemente ou sempre
    'Brancos': (9.7, 5.5, 13.9),
    'Pardos': (32.1, 9.3, 54.9),
    'Pretos': (49.5, 5.9, 93.1),
}
escala_media = {'Brancos': 1.44, 'Pardos': 1.64, 'Pretos': 2.03}   # escala de 1 a 4, dez situações

# Dias (2017), tese de doutorado, Universidade da Califórnia em Berkeley
# Versão em artigo: Dias (2020), Research in Social Stratification and Mobility 65, 100477
# 1.200 currículos fictícios com foto em São Paulo e no Rio, 2014-2015. Mesma foto, tom de pele alterado.
# Um terço dos currículos em cada tom de pele; 768 de mulheres e 432 de homens.
n_curriculos = 1200
n_mulheres_tom = 768 // 3            # 256 currículos de mulheres por tom de pele
n_homens_tom = 432 // 3              # 144 currículos de homens por tom de pele
# Taxa de retorno publicada (%), por tom de pele
retorno_mulheres = {'clara': 18.0, 'média': 16.0, 'escura': 10.6}
retorno_homens = {'clara': 9.0, 'média': 8.3, 'escura': 11.1}
retorno_todos = {'clara': 14.8, 'média': 13.3, 'escura': 10.8}
# Efeito da pele escura contra a clara conforme o nome do candidato (efeito marginal, erro padrão)
nome_classe_baixa = (-0.0627, 0.031)
nome_classe_media = (-0.0166, 0.0312)

# Gerard, Lagos, Severnini & Card (2021), American Economic Review 111(10), 3418-3457
# Números lidos na versão de trabalho de 2018 (IZA DP 11923). RAIS 2002-2014, Sudeste.
# Diferença de salário entre brancos e não brancos (pretos e pardos), em pontos logarítmicos
dif_salario = {'Homens': 18.6, 'Mulheres': 26.1}
parte_empresas = {'Homens': 4.4, 'Mulheres': 7.1}        # vem das empresas: onde a pessoa trabalha e quanto a empresa paga a ela
pagamento_relativo = {'Homens': 1.0, 'Mulheres': 1.5}    # dentro da parte das empresas: pagar menos na mesma empresa
sem_explicacao_pct = {'Homens': 6.0, 'Mulheres': 8.0}    # % da diferença total: alocação que a qualificação não explica

# Sehgal & Svirsky (2024), arXiv 2412.14307. Texto sem revisão por pares.
# Anúncios no Instagram com a mesma foto em pele clara e pele escura. Curtidas e visualizações.
anuncio_br = {'escura': (9588, 71710), 'clara': (9375, 59509)}     # Brasil, réplica exploratória
anuncio_eua = {'escura': (3650, 17635), 'clara': (3880, 16784)}    # Estados Unidos, experimento principal

FATOR = 0.80
aplica_fator_080 = True           # só nas proporções da pesquisa; não nos registros nem nos experimentos

print("=" * 70)
print("  DADOS DA LITERATURA - A COR DA PELE E O TRATAMENTO")
print("=" * 70)
print("\n  Botelho, Madeira & Rangel (2015), versão de trabalho de 2014")
print("  → " + f"{n_alunos:,}".replace(',', '.') + " alunos do oitavo ano em " + f"{n_turmas:,}".replace(',', '.') + " turmas, São Paulo, 2010")
print("\n  Diferença entre alunos negros e brancos da mesma turma:")
print(f"  → Nota (0 a 10), bruta:                      {nota_bruta:+.3f}")
print(f"  → Nota, mesmo desempenho e comportamento:    {nota_controlada[0]:+.3f} (EP {nota_controlada[1]:.3f})")
print(f"  → Nota, mais notas anteriores:               {nota_com_passado[0]:+.3f} (EP {nota_com_passado[1]:.3f})")
print(f"  → Nota, corrigindo erro de medida da prova:  {nota_instrumental[0]:+.3f} (EP {nota_instrumental[1]:.3f})")
print(f"  → Nota para passar, bruta:                   {passar_bruta:+.2f} pp")
print(f"  → Nota para passar, mesmo desempenho:        {passar_controlada[0]:+.2f} pp (EP {passar_controlada[1]:.2f})")
print(f"  → Acima da média da turma, mesmo desempenho: {media_controlada[0]:+.2f} pp (EP {media_controlada[1]:.2f})")
print("\n  Onde a diferença muda (nota):")
print(f"  → Professor não conhecia o aluno: {nao_conhecia[0]:+.3f} (EP {nao_conhecia[1]:.3f})")
print(f"  → Professor já conhecia o aluno:  {ja_conhecia[0]:+.3f} (EP {ja_conhecia[1]:.3f})")
print(f"  → Professor branco:               {prof_branco[0]:+.3f} (EP {prof_branco[1]:.3f})")
print(f"  → Professor preto ou pardo:       {prof_preto_pardo[0]:+.3f} (EP {prof_preto_pardo[1]:.3f})")
print("\n  Pesquisa Mais Dados Mais Saúde (2025), " + f"{n_pesquisa:,}".replace(',', '.') + " respostas online")
print("  Ser tratado com menos respeito, alguma vez (IC 95%):")
for cor, (e, lo, hi) in respeito_alguma_vez.items():
    print(f"  → {cor:<8} {e:.1f}% [{lo:.1f}; {hi:.1f}]")
print("  Ser tratado com menos respeito, com frequência (IC 95%):")
for cor, (e, lo, hi) in respeito_frequente.items():
    print(f"  → {cor:<8} {e:.1f}% [{lo:.1f}; {hi:.1f}]")
print("\n  Dias (2017), " + f"{n_curriculos:,}".replace(',', '.') + " currículos fictícios com foto, São Paulo e Rio")
print("  Taxa de retorno por tom de pele (clara, média, escura):")
print(f"  → Mulheres: {retorno_mulheres['clara']:.1f}%, {retorno_mulheres['média']:.1f}%, {retorno_mulheres['escura']:.1f}%")
print(f"  → Homens:   {retorno_homens['clara']:.1f}%, {retorno_homens['média']:.1f}%, {retorno_homens['escura']:.1f}%")
print(f"  → Pele escura contra clara, nome de classe baixa: {nome_classe_baixa[0]*100:+.1f} pp (EP {nome_classe_baixa[1]*100:.1f})")
print(f"  → Pele escura contra clara, nome de classe média: {nome_classe_media[0]*100:+.1f} pp (EP {nome_classe_media[1]*100:.1f})")
print("\n  Gerard, Lagos, Severnini & Card (2021), versão de trabalho de 2018, RAIS 2002-2014, Sudeste")
for g in dif_salario:
    print(f"  → {g:<9} diferença de salário {dif_salario[g]:.1f} pontos log · {parte_empresas[g]:.1f} vêm das empresas · {sem_explicacao_pct[g]:.0f}% do total sem explicação na qualificação")
print("\n  Sehgal & Svirsky (2024), texto sem revisão por pares. Curtidas por visualização:")
for pais, d in [('Brasil', anuncio_br), ('Estados Unidos', anuncio_eua)]:
    print(f"  → {pais:<15} pele escura {d['escura'][0]/d['escura'][1]*100:.1f}% · pele clara {d['clara'][0]/d['clara'][1]*100:.1f}%")
print(f"\n  Fator ×0,80 aplicado: só nas proporções da pesquisa")
print("=" * 70)

In [ ]:
# --- O MODELO ---
# Contas próprias sobre os números publicados. Semente 42, 200 mil sorteios.

def ic(coef_ep):
    c, ep = coef_ep
    return (c - 1.96 * ep, c + 1.96 * ep)

# ---------- Camada 1: quanto da diferença bruta sobra ----------
especificacoes = [
    ('Mesmo desempenho e comportamento', nota_controlada),
    ('Mais as notas anteriores', nota_com_passado),
    ('Corrigindo o erro de medida da prova', nota_instrumental),
]
sobra = [(rot, c[0] / nota_bruta * 100) for rot, c in especificacoes]
explicado = [(rot, 100 - s) for rot, s in sobra]

# ---------- Camada 2: a diferença em alunos ----------
def por_mil(coef_ep):
    c, ep = coef_ep
    sim = rng.normal(c, ep, N_SIM) * 10            # pontos percentuais → alunos em cada mil
    return -c * 10, -np.percentile(sim, 97.5), -np.percentile(sim, 2.5)

passar_mil = [(rot, por_mil(c)) for rot, c in zip([e[0] for e in especificacoes],
                                                   [passar_controlada, passar_com_passado, passar_instrumental])]
media_mil = [(rot, por_mil(c)) for rot, c in zip([e[0] for e in especificacoes],
                                                  [media_controlada, media_com_passado, media_instrumental])]
# chance de ter nota para passar: observada entre alunos negros e a que teriam sem a diferença
p_sem_diferenca = (p_passar_negros - passar_controlada[0], p_passar_negros - passar_instrumental[0])
sem_nota_obs = 100 - p_passar_negros
aumento_relativo = (-passar_instrumental[0] / (sem_nota_obs + passar_instrumental[0]) * 100,
                    -passar_controlada[0] / (sem_nota_obs + passar_controlada[0]) * 100)

# ---------- Camada 3: o professor que conhece o aluno ----------
dif_conhece = ja_conhecia[0] - nao_conhecia[0]
ep_dif_conhece = np.sqrt(ja_conhecia[1] ** 2 + nao_conhecia[1] ** 2)
z_conhece = dif_conhece / ep_dif_conhece
p_bilateral = 2 * (1 - stats.norm.cdf(abs(z_conhece)))
sim_nao = rng.normal(*nao_conhecia, N_SIM)
sim_ja = rng.normal(*ja_conhecia, N_SIM)
p_menor_quando_conhece = (sim_ja > sim_nao).mean()

dif_cor_prof = prof_preto_pardo[0] - prof_branco[0]
ep_dif_cor = np.sqrt(prof_preto_pardo[1] ** 2 + prof_branco[1] ** 2)
p_cor_prof = 2 * (1 - stats.norm.cdf(abs(dif_cor_prof / ep_dif_cor)))

# ---------- Camada 4: o que as pessoas relatam ----------
def com_fator(t):
    return tuple(v * FATOR for v in t)

def p_maior(a, b):
    # chance de a proporção do grupo a ser maior que a do grupo b, com erro padrão tirado do IC publicado
    sa = rng.normal(a[0], (a[2] - a[1]) / 3.92, N_SIM)
    sb = rng.normal(b[0], (b[2] - b[1]) / 3.92, N_SIM)
    return (sa > sb).mean()

alguma_fator = {cor: com_fator(v) for cor, v in respeito_alguma_vez.items()}
frequente_fator = {cor: com_fator(v) for cor, v in respeito_frequente.items()}
dist_alguma = alguma_fator['Pretos'][0] - alguma_fator['Brancos'][0]
razao_alguma = respeito_alguma_vez['Pretos'][0] / respeito_alguma_vez['Brancos'][0]
p_alguma = p_maior(respeito_alguma_vez['Pretos'], respeito_alguma_vez['Brancos'])
p_frequente = p_maior(respeito_frequente['Pretos'], respeito_frequente['Brancos'])
largura_ic_frequente_pretos = respeito_frequente['Pretos'][2] - respeito_frequente['Pretos'][1]

# ---------- Camada 5: o currículo ----------
def contagem(taxa, n):
    return int(round(taxa / 100 * n))        # retornos reconstruídos a partir da taxa publicada

def sorteia(k, n):
    return rng.beta(k + 0.5, n - k + 0.5, N_SIM)     # distribuição Beta (Jeffreys)

k_m = {t: contagem(v, n_mulheres_tom) for t, v in retorno_mulheres.items()}
k_h = {t: contagem(v, n_homens_tom) for t, v in retorno_homens.items()}
sim_m = {t: sorteia(k, n_mulheres_tom) for t, k in k_m.items()}
sim_h = {t: sorteia(k, n_homens_tom) for t, k in k_h.items()}
dif_m = (sim_m['clara'] - sim_m['escura']) * 100
dif_h = (sim_h['clara'] - sim_h['escura']) * 100
p_mulheres = (dif_m > 0).mean()
p_homens = (dif_h > 0).mean()
ic_dif_m = np.percentile(dif_m, [2.5, 97.5])
ic_dif_h = np.percentile(dif_h, [2.5, 97.5])
ic_m = {t: np.percentile(v, [2.5, 97.5]) * 100 for t, v in sim_m.items()}
ic_h = {t: np.percentile(v, [2.5, 97.5]) * 100 for t, v in sim_h.items()}
curriculos_extra = (retorno_mulheres['clara'] / retorno_mulheres['escura'] - 1) * 100

# o nome do candidato: a diferença é menor com nome de classe média?
sim_baixa = rng.normal(*nome_classe_baixa, N_SIM)
sim_media_cl = rng.normal(*nome_classe_media, N_SIM)
p_menor_classe_media = (sim_media_cl > sim_baixa).mean()

# ---------- Camada 6: a empresa ----------
empresa = {}
for g in dif_salario:
    recebe = 1000 * np.exp(-dif_salario[g] / 100)          # salário do não branco para cada R$ 1.000 do branco
    falta = 1000 - recebe
    fatia_empresas = parte_empresas[g] / dif_salario[g]
    fatia_sem_expl = sem_explicacao_pct[g] / 100
    fatia_relativo = pagamento_relativo[g] / dif_salario[g]
    empresa[g] = dict(recebe=recebe, falta=falta, fatia_empresas=fatia_empresas * 100,
                      reais_empresas=falta * fatia_empresas,
                      reais_sem_expl=falta * fatia_sem_expl,
                      reais_relativo=falta * fatia_relativo,
                      fatia_soma=(fatia_sem_expl + fatia_relativo) * 100)

# ---------- Camada 7: a publicidade ----------
def anuncio(d):
    e = sorteia(*d['escura']); c = sorteia(*d['clara'])
    extra = (c / e - 1) * 100                              # visualizações a mais para o mesmo número de curtidas
    return (d['escura'][0] / d['escura'][1] * 100, d['clara'][0] / d['clara'][1] * 100,
            np.median(extra), *np.percentile(extra, [2.5, 97.5]))
pub_br = anuncio(anuncio_br)
pub_eua = anuncio(anuncio_eua)

print("=" * 70)
print("  MODELO - RESULTADO")
print("=" * 70)
print(f"\n  CAMADA 1 · Quanto da diferença bruta de {abs(nota_bruta):.2f} ponto sobra")
for (rot, c), (_, s) in zip(especificacoes, sobra):
    lo, hi = ic(c)
    print(f"  → {rot:<38} {c[0]:+.3f} [{lo:+.3f}; {hi:+.3f}]  ·  sobra {s:.0f}% da diferença bruta")
print("\n  CAMADA 2 · A diferença em alunos (a cada 1.000 alunos negros)")
print("  Ficam sem nota para passar, com o mesmo desempenho:")
for rot, (m, lo, hi) in passar_mil:
    print(f"  → {rot:<38} {m:.0f} alunos [{lo:.0f}; {hi:.0f}]")
print("  Deixam de ficar acima da média da turma:")
for rot, (m, lo, hi) in media_mil:
    print(f"  → {rot:<38} {m:.0f} alunos [{lo:.0f}; {hi:.0f}]")
print(f"  → Alunos negros com nota para passar: {p_passar_negros:.1f}%  ·  sem a diferença seriam de {p_sem_diferenca[1]:.1f}% a {p_sem_diferenca[0]:.1f}%")
print(f"  → A diferença aumenta em {aumento_relativo[0]:.0f}% a {aumento_relativo[1]:.0f}% o grupo que fica sem nota para passar")
print("\n  CAMADA 3 · O professor que conhece o aluno")
print(f"  → Não conhecia: {nao_conhecia[0]:+.3f} [{ic(nao_conhecia)[0]:+.3f}; {ic(nao_conhecia)[1]:+.3f}]")
print(f"  → Já conhecia:  {ja_conhecia[0]:+.3f} [{ic(ja_conhecia)[0]:+.3f}; {ic(ja_conhecia)[1]:+.3f}]")
print(f"  → Diferença entre os dois: {dif_conhece:+.3f} (EP {ep_dif_conhece:.3f}), z = {z_conhece:.2f}, p = {p_bilateral:.2f}")
print(f"  → P(a diferença ser menor quando o professor conhece o aluno): {p_menor_quando_conhece*100:.0f}%")
print(f"  → Professor branco contra professor preto ou pardo: diferença de {dif_cor_prof:+.3f}, p = {p_cor_prof:.2f}")
print("\n  CAMADA 4 · O que as pessoas relatam (com o fator ×0,80, e o número original ao lado)")
print("  Ser tratado com menos respeito, alguma vez:")
for cor in respeito_alguma_vez:
    e, lo, hi = alguma_fator[cor]
    print(f"  → {cor:<8} {e:.0f}% [{lo:.0f}; {hi:.0f}]   (sem o fator {respeito_alguma_vez[cor][0]:.1f}%)")
print(f"  → Distância entre pretos e brancos: {dist_alguma:.0f} pontos percentuais  ·  razão {razao_alguma:.1f}")
print(f"  → P(pretos acima de brancos): {p_alguma*100:.1f}%")
print("  Ser tratado com menos respeito, com frequência:")
for cor in respeito_frequente:
    e, lo, hi = frequente_fator[cor]
    print(f"  → {cor:<8} {e:.0f}% [{lo:.0f}; {hi:.0f}]   (sem o fator {respeito_frequente[cor][0]:.1f}%)")
print(f"  → P(pretos acima de brancos): {p_frequente*100:.0f}%  ·  largura do intervalo dos pretos: {largura_ic_frequente_pretos:.0f} pontos")
print("\n  CAMADA 5 · O currículo (retornos a cada 100 currículos iguais, só o tom de pele muda)")
for t in ['clara', 'média', 'escura']:
    print(f"  → Mulheres, pele {t:<7} {retorno_mulheres[t]:.1f}% [{ic_m[t][0]:.1f}; {ic_m[t][1]:.1f}]   ({k_m[t]} de {n_mulheres_tom})")
for t in ['clara', 'média', 'escura']:
    print(f"  → Homens, pele {t:<7}   {retorno_homens[t]:.1f}% [{ic_h[t][0]:.1f}; {ic_h[t][1]:.1f}]   ({k_h[t]} de {n_homens_tom})")
print(f"  → Mulheres, clara menos escura: {np.median(dif_m):.1f} pp [{ic_dif_m[0]:.1f}; {ic_dif_m[1]:.1f}]  ·  P(retorno menor com pele escura) = {p_mulheres*100:.1f}%")
print(f"  → Homens, clara menos escura:   {np.median(dif_h):.1f} pp [{ic_dif_h[0]:.1f}; {ic_dif_h[1]:.1f}]  ·  P(retorno menor com pele escura) = {p_homens*100:.0f}%")
print(f"  → Para o mesmo número de retornos, a mulher de pele escura envia {curriculos_extra:.0f}% mais currículos")
print(f"  → Nome de classe baixa: {nome_classe_baixa[0]*100:+.1f} pp  ·  nome de classe média: {nome_classe_media[0]*100:+.1f} pp")
print(f"  → P(a diferença ser menor com nome de classe média): {p_menor_classe_media*100:.0f}%")
print("\n  CAMADA 6 · A empresa (para cada R$ 1.000 de salário de um trabalhador branco)")
for g, e in empresa.items():
    print(f"  → {g:<9} o não branco recebe R$ {e['recebe']:.0f}  ·  faltam R$ {e['falta']:.0f}")
    print(f"     R$ {e['reais_empresas']:.0f} ({e['fatia_empresas']:.0f}%) vêm das empresas (onde a pessoa trabalha e quanto a empresa paga a ela)")
    print(f"     desses, R$ {e['reais_sem_expl']:.0f} são alocação que a qualificação não explica  ·  R$ {e['reais_relativo']:.0f} são pagamento menor na mesma empresa")
    print(f"     somadas, as duas parcelas são {e['fatia_soma']:.0f}% da diferença")
print("\n  CAMADA 7 · A publicidade (curtidas por visualização)")
for pais, p in [('Brasil', pub_br), ('Estados Unidos', pub_eua)]:
    print(f"  → {pais:<15} pele escura {p[0]:.1f}% · pele clara {p[1]:.1f}%  ·  {p[2]:.0f}% mais visualizações para as mesmas curtidas [{p[3]:.0f}; {p[4]:.0f}]")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO - GRÁFICOS SEPARADOS ---

# ── GRÁFICO 1 · O que as pessoas relatam ──
fig1, ax1 = plt.subplots(figsize=(12, 8))
cores_g = ['Brancos', 'Pardos', 'Pretos']
x = np.arange(3)
larg = 0.36
for desloc, dados, cor, rot in [(-larg / 2, alguma_fator, VERDE, 'alguma vez'), (larg / 2, frequente_fator, VERMELHO, 'com frequência')]:
    v = [dados[c][0] for c in cores_g]
    err = [[dados[c][0] - dados[c][1] for c in cores_g], [dados[c][2] - dados[c][0] for c in cores_g]]
    ax1.bar(x + desloc, v, larg, color=cor, alpha=0.85, label=rot)
    ax1.errorbar(x + desloc, v, yerr=err, fmt='none', color='#222', capsize=6, lw=1.8)
    for xi, c in zip(x + desloc, cores_g):
        ax1.text(xi, dados[c][2] + 1.5, f'{dados[c][0]:.0f}%', ha='center', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(cores_g, fontsize=13)
ax1.set_ylim(0, 95)
ax1.set_ylabel('Dizem ser tratados com menos respeito (%, com o fator ×0,80)', fontsize=12)
ax1.set_title('O que as pessoas relatam: ser tratado com menos respeito, por cor\n'
              'Pesquisa online com 2.458 respostas, 2024. Barras com IC 95%', fontsize=14, pad=15)
ax1.legend(fontsize=12, loc='upper left')
plt.figtext(0.5, 0.01, 'Fonte: Mais Dados Mais Saúde (Vital Strategies, Umane, UFPel e Ministério da Igualdade Racial), 2025 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-160-grafico-01-relato.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 · A escola ──
fig2, ax2 = plt.subplots(figsize=(12, 8))
rot2 = ['Mesmo desempenho\ne comportamento', 'Mais as notas\nanteriores', 'Corrigindo o erro\nde medida da prova']
x = np.arange(3)
larg = 0.36
v_p = [m for _, (m, lo, hi) in passar_mil]
e_p = [[m - lo for _, (m, lo, hi) in passar_mil], [hi - m for _, (m, lo, hi) in passar_mil]]
v_m = [m for _, (m, lo, hi) in media_mil]
e_m = [[m - lo for _, (m, lo, hi) in media_mil], [hi - m for _, (m, lo, hi) in media_mil]]
ax2.bar(x - larg / 2, v_p, larg, color=VERMELHO, alpha=0.85, label='ficam sem nota para passar')
ax2.bar(x + larg / 2, v_m, larg, color=DOURADO, alpha=0.85, label='deixam de ficar acima da média da turma')
ax2.errorbar(x - larg / 2, v_p, yerr=e_p, fmt='none', color='#222', capsize=6, lw=1.8)
ax2.errorbar(x + larg / 2, v_m, yerr=e_m, fmt='none', color='#222', capsize=6, lw=1.8)
for xi, v, (_, (m, lo, hi)) in zip(x - larg / 2, v_p, passar_mil):
    ax2.text(xi, hi + 0.8, f'{v:.0f}', ha='center', fontsize=15, fontweight='bold')
for xi, v, (_, (m, lo, hi)) in zip(x + larg / 2, v_m, media_mil):
    ax2.text(xi, hi + 0.8, f'{v:.0f}', ha='center', fontsize=15, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(rot2, fontsize=12)
ax2.set_ylim(0, 36)
ax2.set_ylabel('Alunos em cada 1.000 alunos negros', fontsize=12)
ax2.set_title('Na escola: com o mesmo desempenho, quantos alunos negros em cada mil ficam para trás na nota\n'
              'Três formas de igualar o desempenho, da menos à mais exigente', fontsize=14, pad=15)
ax2.legend(fontsize=11, loc='upper right')
plt.figtext(0.5, 0.01, 'Modelo próprio sobre os coeficientes de Botelho, Madeira & Rangel, 2015 (versão de trabalho de 2014). Barras com IC 95% | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-160-grafico-02-escola.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 · O currículo ──
fig3, ax3 = plt.subplots(figsize=(12, 8))
tons = ['clara', 'média', 'escura']
x = np.arange(3)
larg = 0.36
for desloc, taxa, icg, cor, rot in [(-larg / 2, retorno_mulheres, ic_m, VERMELHO, 'currículos de mulheres'),
                                     (larg / 2, retorno_homens, ic_h, CINZA, 'currículos de homens')]:
    v = [taxa[t] for t in tons]
    err = [[taxa[t] - icg[t][0] for t in tons], [icg[t][1] - taxa[t] for t in tons]]
    ax3.bar(x + desloc, v, larg, color=cor, alpha=0.85, label=rot)
    ax3.errorbar(x + desloc, v, yerr=err, fmt='none', color='#222', capsize=6, lw=1.8)
    for xi, t in zip(x + desloc, tons):
        ax3.text(xi, icg[t][1] + 0.6, f'{taxa[t]:.0f}'.replace('.', ','), ha='center', fontsize=15, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(['Foto com pele clara', 'Foto com pele média', 'Foto com pele escura'], fontsize=13)
ax3.set_ylim(0, 30)
ax3.set_ylabel('Retornos a cada 100 currículos enviados', fontsize=12)
ax3.set_title('No currículo: a mesma foto e o mesmo currículo, com o tom de pele alterado\n'
              f'Entre as mulheres, P(retorno menor com pele escura) = {p_mulheres*100:.0f}%', fontsize=14, pad=15)
ax3.legend(fontsize=12, loc='upper right')
plt.figtext(0.5, 0.01, 'Fonte: Dias, 2017 (1.200 currículos, São Paulo e Rio). Intervalos de 95% calculados aqui | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-160-grafico-03-curriculo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ── GRÁFICO 4 · Onde a diferença diminui ──
fig4, (axa, axb) = plt.subplots(1, 2, figsize=(14, 7))
paineis = [
    (axa, [('Professor não\nconhecia o aluno', nao_conhecia, VERMELHO), ('Professor já\nconhecia o aluno', ja_conhecia, VERDE)], 1,
     'Na escola\ndiferença na nota (pontos)', f'P(menor quando conhece) = {p_menor_quando_conhece*100:.0f}%'),
    (axb, [('Nome associado\na classe baixa', nome_classe_baixa, VERMELHO), ('Nome associado\na classe média', nome_classe_media, VERDE)], 100,
     'No currículo\ndiferença no retorno, pele escura contra clara (pp)', f'P(menor com nome de classe média) = {p_menor_classe_media*100:.0f}%'),
]
for ax, itens, escala, titulo, nota in paineis:
    for i, (rot, c, cor) in enumerate(itens):
        ax.bar(i, c[0] * escala, 0.5, color=cor, alpha=0.85)
        ax.errorbar(i, c[0] * escala, yerr=1.96 * c[1] * escala, color='#222', capsize=7, lw=1.8)
        casas = 3 if escala == 1 else 1
        ax.text(i + 0.3, c[0] * escala, f'{c[0]*escala:+.{casas}f}'.replace('.', ','), va='center', fontsize=14, fontweight='bold')
    ax.axhline(0, color='#222', lw=1.3)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([it[0] for it in itens], fontsize=12)
    ax.set_xlim(-0.6, 1.8)
    ax.set_ylim((-0.07, 0.04) if escala == 1 else (-14, 9))
    ax.set_title(titulo, fontsize=13, pad=12)
    ax.text(0.5, 0.93, nota, transform=ax.transAxes, ha='center', fontsize=12, color='#333',
            bbox=dict(facecolor='white', edgecolor='#bbb', boxstyle='round,pad=0.4'))
fig4.suptitle('A diferença diminui quando quem avalia tem mais informação sobre a pessoa', fontsize=15)
plt.figtext(0.5, 0.01, 'Fontes: Botelho, Madeira & Rangel, 2015 (versão de trabalho) · Dias, 2017. Barras com IC 95% | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('dia-160-grafico-04-onde-diminui.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# ── GRÁFICO 5 · A empresa ──
fig5, ax5 = plt.subplots(figsize=(12, 8))
grupos5 = list(empresa.keys())
y = np.arange(len(grupos5))[::-1]
for yi, g in zip(y, grupos5):
    e = empresa[g]
    partes = [(e['reais_sem_expl'] + e['reais_relativo'], VERMELHO),
              (e['reais_empresas'] - e['reais_sem_expl'] - e['reais_relativo'], DOURADO),
              (e['falta'] - e['reais_empresas'], CINZA)]
    esq = 0
    for valor, cor in partes:
        ax5.barh(yi, valor, left=esq, color=cor, alpha=0.85, height=0.5)
        ax5.text(esq + valor / 2, yi, f'R$ {valor:.0f}', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
        esq += valor
    ax5.text(esq + 4, yi, f'faltam R$ {e["falta"]:.0f}', va='center', fontsize=13, fontweight='bold')
ax5.set_yticks(y)
ax5.set_yticklabels(grupos5, fontsize=13)
ax5.set_xlim(0, 300)
ax5.set_ylim(-0.6, 1.9)
ax5.set_xlabel('Reais a menos para cada R$ 1.000 de salário de um trabalhador branco', fontsize=12)
ax5.set_title('Na empresa: de onde vem a diferença de salário entre brancos e não brancos\n'
              'Emprego formal no Sudeste, 2002 a 2014', fontsize=14, pad=15)
from matplotlib.patches import Patch
ax5.legend(handles=[Patch(color=VERMELHO, alpha=0.85, label='empresa: parte que a qualificação não explica'),
                    Patch(color=DOURADO, alpha=0.85, label='empresa: parte ligada à qualificação'),
                    Patch(color=CINZA, alpha=0.85, label='características do trabalhador e outros fatores')],
           fontsize=11, loc='upper right')
plt.figtext(0.5, 0.01, 'Modelo próprio sobre Gerard, Lagos, Severnini & Card, 2021 (versão de trabalho de 2018) | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-160-grafico-05-empresa.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

### 💡 O Insight

**De cada 100 currículos iguais de mulheres, 18 tiveram retorno com a foto de pele clara e 11 com a de pele escura.**

O currículo era o mesmo e a foto era a mesma, com o tom de pele alterado. A chance de o retorno ser de fato menor com pele escura é de 99%. Entre os homens, o experimento não encontrou diferença.

Nos outros três lugares a direção é a mesma.

Na escola, com o mesmo desempenho na prova sem nome, de 6 a 10 alunos negros em cada mil ficam sem nota para passar.

Na empresa, para cada R$ 1.000 que um homem branco recebe, um homem não branco recebe R$ 830. Dos R$ 170 que faltam, cerca de R$ 19 vêm de práticas das empresas que a qualificação não explica. Entre as mulheres são R$ 32 de R$ 230.

Na publicidade, o mesmo anúncio com pele escura recebeu 13 curtidas a cada 100 visualizações no Brasil, contra 16 com pele clara. É o dado mais frágil dos quatro.

O relato das pessoas é bem maior do que qualquer uma dessas medidas. Na pesquisa, 74% dos pretos dizem já ter sido tratados com menos respeito, contra 37% dos brancos, já com o fator ×0,80. As medidas não se somam. A pesquisa mostra o que as pessoas vivem em todas as situações do dia. Cada estudo mostra o que sobra numa situação só, depois de retirar tudo o que tem outra explicação.

Dois estudos mostram também onde a diferença diminui. Na escola, ela fica perto de zero quando o professor já conhecia o aluno. No currículo, ela é de 6 pontos percentuais com nome associado a classe baixa e de 2 com nome associado a classe média. Nos dois casos, quem avalia tinha mais informação sobre a pessoa.

A pergunta que fica:

*Em quantas decisões do seu dia você avalia alguém sem conhecer a pessoa?*

---

### ⚠️ Limitações do Modelo

- As quatro medidas vêm de populações, anos e métodos diferentes. Elas não devem ser somadas nem comparadas ponto a ponto. A pergunta do título não tem um número único como resposta.
- Só o experimento dos currículos sorteia a cor, e só ele permite falar em causa. A escola e a empresa são comparações com controles, e algum fator não observado pode restar.
- **Currículos.** Os números são os da tese (1.200 currículos). O artigo publicado em 2020 informa 820 currículos e taxa de retorno de 10,7%, contra 12,9% na tese, e só o seu resumo foi lido. A tese publica as taxas e não as contagens por grupo, então as contagens foram reconstruídas supondo um terço dos currículos em cada tom de pele, e os intervalos são nossos.
- **Currículos.** O efeito aparece só entre as mulheres. Entre os homens a amostra é menor (432 currículos) e o intervalo é largo, o que não permite afirmar que o efeito não existe.
- **Currículos.** A diferença entre nomes de classe baixa e de classe média tem 85% de chance de existir pela nossa conta. É um indício, não uma prova. A tese cobre vagas de entrada (recepção, vendas, auxiliar administrativo) em duas cidades.
- **Currículos.** A tese traz uma inconsistência entre texto e tabela sobre a significância do efeito no conjunto de homens e mulheres. O resultado usado aqui é o das mulheres, que é consistente nos dois.
- **Escola.** Os números vêm da versão de trabalho de 2014. O estudo cobre só matemática, só o oitavo ano, só 2010 e só a rede estadual de São Paulo. A versão lida compara alunos que se declararam negros com os que se declararam brancos, e os pardos não aparecem em separado.
- **Escola.** A diferença entre professores que conheciam e que não conheciam o aluno tem p = 0,10 no teste direto. "Discriminação estatística" é a interpretação dos autores, e o desenho não observa o que o professor pensa.
- **Empresa.** Os números vêm da versão de trabalho de 2018, que não publica erro padrão para a decomposição, e por isso essa camada não tem intervalo. As diferenças estão em pontos logarítmicos e foram convertidas em reais por nós. Os autores dizem que a parte sem explicação é compatível com contratação excludente, e não a testam diretamente.
- **Empresa.** O estudo cobre só o emprego formal do Sudeste e reúne pretos e pardos em um grupo. A parcela cinza do gráfico (características do trabalhador) inclui escolaridade e experiência, que têm as suas próprias causas, e não deve ser lida como diferença justa.
- **Publicidade.** O texto não passou por revisão por pares. A parte brasileira é uma réplica exploratória, e os autores avisam que os grupos de comparação não ficaram equilibrados, o que dificulta a leitura. O percentual de visualizações a mais é conta nossa sobre as contagens totais. O intervalo trata cada visualização como independente e por isso é mais estreito do que deveria.
- **Pesquisa.** É online, com recrutamento por redes sociais, taxa de resposta de 4,7% e amostra não probabilística. Na versão "com frequência", o intervalo dos pretos vai de 5% a 74% com o fator, e por isso o texto usa a versão "alguma vez".
- O fator ×0,80 é uma convenção editorial conservadora do projeto, e não uma correção científica. O número sem o fator está no notebook.
- As fontes usam classificações diferentes (cor declarada pelo aluno, tom de pele da foto, cor registrada pelo empregador, cor declarada na pesquisa). O texto mantém a classificação de cada uma.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*